# Building and Learning (Pseudo)Metrics -- Fourier Case

A metric is a notion of distance generalized to arbitrary topological spaces. It has four key properties:
1. *Nonnegativity*: $d(x, y) \geq 0$
2. *Uniqueness*: if $d(x,y) = 0$, then $x=y$
3. *Symmetry*: $d(x,y) = d(y,x)$
4. *Triangle Identity*: $d(x, z) \leq d(x, y) + d(y, z)$

A _pseudometric_ lacks the uniqueness property: It has $x \neq y$ with $d(x,y) = 0$. 

For neural localization, we need to compare two spaces: locations and sensory percepts. In general, the representation spaces for these two spaces don't naturally work with the Euclidean metric, especially for locations as described by grid cell codes. So we'll first define an explicit metric for the Fourier space of grid cells in this notebook, and then consider how to learn a metric that is useful for comparing sensory percepts that could also be applied to location space in another.

We need this metrics to build spatial memories for non-Euclidean spaces to support TEM.

## Fourier Location Encoding

Grid cells in $d$ dimensions (usually $d=2$) fire with a rate described by
$$
r_j(x) = \left(\sum_{i=1}^{d+1} A_{ij} \cos\left(\vphantom{x^x}k_{i}\cdot x + \phi_{ij}\right)\right)_+
$$
when the organism is at position $x$, where $A_{ij}$ is a fixed amplitude and $k_{i}$ is a wave vector, one of $d + 1$ unit vectors whose convex hull forms the simplex. In 2-D, these vectors start at the origin and are radially separated by $120^\circ$. In 3-D, four wavevectors are needed, pointing the the vertices of a triangular pyramid.

We can model these cells via an assemblage of column vectors having the form
$$
\eta_{ij}(x) = \left(\begin{array}{c}
    \cos (\alpha_j k_i\cdot x + \phi_j) \\
    \sin (\alpha_j k_i\cdot x + \phi_j)
\end{array}\right)
$$
where the $\{\phi_j\}$ are chosen to cover the interval $[0, 2\pi)$. Sufficiently wide choices of $\phi_j$ will cover the full circle.

Grid cell responses are periodic as the animal moves around its environment, and thanks to the identity
$$
\cos a \cos b  + \sin a \sin b = \cos (a - b) = \cos (b - a)
$$
we can arrive a relationship between displacement in physical space and rotation in Fourier space. To do this, we form a matrix $K$ whose rows range over $k_i$, so that $K$ is a $(d+1) \times d$ matrix. The rows of $K$ span $\mathbb{R}^d$, so the $d \times d$ matrix $K^TK$ is invertible, and the $d \times (d+1)$ pseudoinverse of this matrix is $K^\dagger = (K^TK)^{-1}K^T$ has $K^\dagger K = I$. Then we have
$$
\left(\vphantom{x^{x^x}}\eta_{ij}(x), \,\eta_{ij}(x + \Delta x)\right) \,\,=\,\, 
\cos(\alpha_j k_i\cdot \Delta x)
\quad\quad\text{whence}\quad\quad \forall n_j \in \mathbb{Z}^{d+1},\quad
\alpha_j K \Delta x = 2\pi n_j + \mathrm{arccos} \left(\vphantom{x^{x^{x^x}}}\Eta_{j}(x)^T\,\Eta_{j}(x + \Delta x)\right)
$$
where $\Eta_j(x)$ is the $(d+1) \times 2$ matrix whose columns are $\eta_{ij}(x)$.
Solving for $\Delta x$, we find
$$
\forall j\,\,
\exists n_j \in \mathbb{Z}^{d+1}\quad\text{s.t.}\quad\quad
\Delta x = \frac{2\pi K^\dagger\,n_j}{\alpha_j} + \frac{K^\dagger}{\alpha_j} \Delta \theta_{j}\quad\quad\text{given}\quad\quad\Delta\theta_{j} = \mathrm{arccos} \left(\vphantom{x^{x^{x^x}}}\Eta_{j}(x)^T\,\Eta_{j}(x + \Delta x)\right)
$$
With sufficient and carefully chosen $(\alpha_j, \phi_j)$, these modular constraints can be solved for a wide range of $\Delta x$, and $\Delta x$ can be identified exactly on this range.

So the firing rates of an assemblage of grid cells with different phases $\phi_j$ can be stably and reliably inverted to retrieve the physical location $x$.

## From One Location to the Next

We need a function to update locations based on actions, $\ell_t = h(\ell_{t-1}, a_{t-1})$.

We will assume our actions $a_t$ has an impact on location as translation in physical space. If we assume Fourier location codes (like $\eta_{ij}$ above), then a displacement $\Delta x$ becomes a block-wise rotation of each $\eta_{ij}$. Recall that a rotation by angle $\theta$ is 
$$
R(\theta) = \left[\begin{array}{cc}
    \cos \theta & -\sin \theta \\
    \sin \theta & \cos \theta
\end{array}\right],
$$
so notating $\eta_{ij} = \left(\begin{array}{c}x\\y\end{array}\right)$, we have
$$
R(\theta_{ij})\eta_{ij} = \left(\begin{array}{c}
x\cos\theta_{ij} - y\sin\theta_{ij} \\
x\sin\theta_{ij} + y\cos\theta_{ij}
\end{array}\right).
$$

Beyond this rotational update, the Fourier coding can remain implicit in our location-learning scheme. Thus we do not need to know $\alpha_j$ or $\phi_{ij}$, but instead we allow $\theta_{ij}$ to differ for each $ij$. Then we can learn a function $g_\phi$ with parameters $\phi$ and apply it as a block-diagonal matrix multiply as follows:
$$
\theta_{ij} = g_\phi(a_{t-1})
\quad\quad\text{and}\quad\quad
\ell_t = \left[\begin{array}{ccc}
R(\theta_{00}) & \ldots & 0 \\
\vdots & \ddots & \vdots \\
0 & \ldots & R(\theta_{IJ})
\end{array}\right] \ell_{t-1}
$$
This treats $\ell_t$ as a sequence of pairs of components $\eta_{ij}$ under some enumeration of $ij$. The following code implements this scheme more efficiently, without assembling the full block-diagonal matrix. Each application extends the input sequence by one.

## Making Location Codes Physical

A location code represents a physical point. A system of location codes has to represent different points consistently. That restricts the codes that can simultaneously be valid. 

When it comes to changes in location, simply updating the location codes by rotation does not guarantee a unique solution for the displacement $\Delta x$; different $\eta_{ij}$ can yield different displacements. Basically, these location codes have extra flexibility that might lead to a suboptimal spatial representation. The same position in real space could have many different codes, which will prevent us from being able to generate an accurate map of local space, or to use the map to go to areas of interest or to judge what objects are close to each other.

Note that in our geometric decoder, we have calculated $\Delta\theta_{ij} = g_\phi(a_t)$. In general, our location codes should adhere to
$$
\Delta x = \frac{2\pi K^\dagger\,n_j}{\alpha_j} + \frac{1}{\alpha_j} K^\dagger\,\,\Delta \theta_{j}
\quad\quad\text{for some }n_j \in \mathbb{Z}^{d+1}, \,\,\alpha_j \in \mathbb{R}
$$
based on the formulae above. Our problem is that different $j$ will give us different values of $\Delta x$; we can use $\Delta x_j$ to represent the value given for the matrix $\Eta_{j}$, which has size $(d+1) \times 2$.

Hence we have an estimator for a random variable $\Delta X = \frac{1}{J}\sum_j \Delta x_j$, and we can minimize the variance of this estimator
$$
\mathcal{L}_{\text{consistency}} \,=\, \mathrm{Var}^2\left[\Delta X\right]\,\,=\,\, \frac{1}{J-1}\sum_j \left\|\Delta x_j - \Delta X\right\|^2
$$
but first we must solve for $\Delta x_j$. To do this, we first canonicalize $\Delta\theta_j \in [-\pi, \pi)$ and compute $\overline{\Delta x}_j = \frac{1}{\alpha_j} K^\dagger\,\,\Delta \theta_{j}$ and then note that 
$$
\Delta x_j \in \overline{\Delta x}_j + \Lambda_j
\quad\quad\text{for}\quad\quad
\Lambda_j \,\,=\,\, \frac{2\pi}{\alpha_j} K^\dagger\,\,\mathbb{Z}^{d+1} \,\,=\,\, \left\{\left.\frac{2\pi K^\dagger\,n_j}{\alpha_j}\,\right\vert\,n_j \in \mathbb{Z}^{d+1}\right\}.
$$

Our problem is that we have too many variables $(d+1)$ for too few constraints ($d$). We need to eliminate a constraint, and because we chose $K$ to be the vertices of the $d+1$ simplex, the structure of our problem lets us do that. We can choose a "basis" $B_j$ that generates $\Lambda_j$ as follows. Let 
$$
U = \left[e_1 - e_{d+1}, \ldots e_d - e_{d+1}\right]  \in \mathbb{Z}^{(d+1)\times d}
$$
where the $e_i$ are the standard basis vectors of $\mathbb{R}^{d+1}$, that is, $e_i$ is the $i^{th}$ row or column of the identity matrix $I_{d+1}$. For any $n \in \mathbb{Z}^{d+1}$, we then have that
$$
\exists m_j \in \mathbb{Z}^d\quad\exists b_j \in \mathbb{Z} \quad \text{s.t.}\quad n_j = Um_j + b_j\mathbf{1}
\quad\quad\text{wherefore}\quad\quad
K^\dagger\, n_j \,\,=\,\, K^\dagger\,Um + b_j\,K^\dagger\,\mathbf{1}\,\,=\,\, K^\dagger\,Um_j
$$
because $K^\dagger\,\mathbf{1} = 0$, which is a consequence of choosing $K$ to be the vertices of the simplex. Now, we define
$$
B_j = \frac{2\pi}{\alpha_j} K^\dagger\,U \quad\in\,\mathbb{R}^{d\times d}
\quad\quad\text{which yields}\quad\quad
\Delta x_j \in \overline{\Delta x}_j + B_j \mathbb{Z}^d
$$
which reduces the number of constraints by one, allowing us to find solutions.

Let's compute $K$ and $B_j$ for all $j$.

In [1]:
import math
import torch


def make_lattice_basis(alphas, dim: int = 2):
    """
    Construct lattice basis matrices B_j for each grid module j in d dimensions.

    Args:
        alphas: 1D tensor of shape (J,) with spatial frequencies α_j.
        dim: spatial dimension d (>=1).

    Returns:
        B: tensor of shape (J, d, d), where B[j] is the lattice basis B_j.
        K: tensor of shape (d+1, d) with simplex directions as rows.
        K_dagger: tensor of shape (d, d+1) with pseudoinverse of K, returned for convenience.
    """
    if dim < 1:
        raise ValueError("dimension must be >= 1")

    # Ensure alphas is a tensor of shape (J,)
    dtype = alphas.dtype
    device = alphas.device
    J = alphas.shape[0]

    # 1. Build a basis U for the null subspace of the simplex K
    U = torch.cat([torch.eye(dim, dtype=dtype, device=device), -torch.ones(1, dim, dtype=dtype, device=device)], dim=0)

    # 2. Build a regular simplex in R^(d+1) and normalize
    Q, _ = torch.linalg.qr(U, mode="reduced")  # Q: (d+1, d)
    V = torch.eye(dim + 1, dtype=dtype, device=device) - (1.0 / (dim + 1))
    K = V @ Q  # (d+1, d)
    K = K / K.norm(dim=1, keepdim=True)  # each row is now unit length

    # Pseudoinverse of K: K^† ∈ R^{d×(d+1)}
    K_dagger = torch.linalg.pinv(K)

    # Base (unscaled) lattice generator: shape (d, d)
    base = K_dagger @ U

    # 3. Scale by 2π / α_j for each module
    scale = (2.0 * math.pi) / alphas.view(J, 1, 1)  # (J, 1, 1)
    B = scale * base[None, ...]                  # (J, d, d)

    return B, K, K_dagger

To solve for $\Delta_j$, we will rely on the $\alpha_j$ being sorted from coarsest ($\alpha_j$ small = long wavelength) to finest. We will assume this as a matter of initialization for $\alpha_j$, which will be fixed and not learned. Let's look at our initialization of $\alpha_j$ for the decoding process.

The choice of $\alpha_j$ will determine the range of $\Delta x$ that can be recovered. So we will use a `scale` parameter ($s$). This parameter is, in effect, the maximum norm of displacement caused by any action. Hence actions determine the scale of the space.

Rather than generating the $\alpha_j$ directly, we will instead focus on the wavelength $\lambda_j = {2\pi}/{\alpha_j}$. Since our $\alpha_j$ will be sorted, we want $\lambda_0 = 2s$. This will allow us to recover actions with norms in the range $[-s, s)$. From there, each successive $j$ will use a smaller and smaller wavelength in a geometric pattern. The `ratio` ($\rho$) of reduction is a parameter to our generator, and we will have $\lambda_j = 2s\rho^{-j}$, whence $\alpha_j = \pi\rho^j / s$. 

In [2]:
import math
import torch


def make_alphas(location_dim: int, dim: int = 2, scale: float = 10.0,
                ratio: float = math.sqrt(2.0), dtype=torch.get_default_dtype(), device=torch.device("cpu")) -> torch.Tensor:
    """
    Choose module spatial frequencies alpha_j given a location code size and
    a target "safe" displacement scale.

    Args:
        location_dim: int, total number of phase channels = J * (d+1).
        dim: spatial dimension d.
        scale: radius such that for ||Δx|| < scale, the coarsest module
               is unambiguous (λ_0/2 ≈ scale).
        ratio: geometric ratio between successive periods (default √2).

    Returns:
        alphas: (J,) tensor of spatial frequencies α_j, with j=0 coarsest.
    """
    num_dirs = dim + 1
    assert location_dim % (2 * num_dirs) == 0,  f"location_dim={location_dim} must be divisible by 2*(dimension+1)={2*num_dirs}"
    J = location_dim // num_dirs // 2  # number of modules
    j_idx = torch.arange(J, dtype=dtype, device=device)
    return (math.pi / scale) * ratio ** j_idx

Finally, to solve for $\Delta_j$, we iterate over $j$, choosing $n_0 = 0$ as our initial estimate on the assumption that $\|\Delta x\| < s$ for our `scale` parameter $s$. That is, $\Delta x_0 = \overline{\Delta x}_0$. Then, we choose the value for $n_j$ that minimizes the error $\|\Delta x_0 - \Delta x_j\|$ by rounding. That is, from $\Delta x_j = \overline{\Delta x}_j + B_jn_j$ we equate $\Delta x_j = \Delta x_0$, which leads to
$$
B_j n_j = \Delta x_0 - \overline{\Delta x}_j \,\,\equiv\,\, \delta_j
$$
Now the matrix formula $B_j z = \delta_j$ can be solved for $z \in \mathbb{R}^d$, and we can estimate $n_j = \mathrm{round}(z)$, allowing us to compute $\Delta x_j$.


In [66]:
def solve_for_deltas(delta_thetas: torch.Tensor, K_dagger: torch.Tensor, lattice_basis: torch.Tensor, alphas: torch.Tensor):
    d, dplus = K_dagger.shape
    J, _, _ = lattice_basis.shape
    shape = delta_thetas.shape[:-1] + (J, dplus)
    delta_thetas = delta_thetas.view(shape)[..., None]
    while K_dagger.ndim < delta_thetas.ndim:
        K_dagger = K_dagger[None, ...]
    displacement_base = (K_dagger @ delta_thetas).view(shape[:-1] + (d,)) / alphas[None, :, None]
    reference_displacement = displacement_base[..., 0, :]
    errors = reference_displacement[..., None, :] - displacement_base
    lattice_basis = lattice_basis.view(J, d, d)
    while lattice_basis.ndim < errors.ndim + 1:
        lattice_basis = lattice_basis[None, ...]
    offsets = torch.linalg.solve(lattice_basis.float(), errors[..., None].float()).round().to(delta_thetas.dtype)
    deltas = displacement_base + (lattice_basis @ offsets).squeeze(-1)
    return deltas.view(shape[:-1] + (d,))


Now we can check these functions for accuracy:

In [18]:
J = 20
d = 2

location_dim = J * (d+1) * 2
alphas = make_alphas(location_dim, d)
print(f"alphas: {alphas.detach().cpu().numpy().tolist()}")

lattice_basis, K, K_dagger = make_lattice_basis(alphas, d)
print(f"K: {K.detach().cpu().numpy().tolist()}")
print(f"K_dagger: {K_dagger.detach().cpu().numpy().tolist()}")
assert torch.allclose(K.sum(dim=0), torch.zeros(d))

phis = torch.linspace(0, 2*math.pi, J)

x0 = torch.randn(2, 5, d)
xf = torch.randn(2, 5, d)
deltas = xf - x0

theta0 = alphas[None, None, :, None, None] * (K[None, None, ...] @ x0.view(2, 5, d, 1))[:, :, None, :, :] + phis[None, None, :, None, None]
thetaf = alphas[None, None, :, None, None] * (K[None, None, ...] @ xf.view(2, 5, d, 1))[:, :, None, :, :] + phis[None, None, :, None, None]

print(theta0.shape)

delta_thetas = (thetaf - theta0).view(2, 5, -1)

delta_thetas = delta_thetas.view(2, 5, -1)
deltas_estimate = solve_for_deltas(delta_thetas, K_dagger, lattice_basis, alphas)

print("AVERAGE ERROR OF ESTIMATED DELTA FROM TRUE DELTA: ", torch.norm(deltas[:, :, None, :] - deltas_estimate, dim=-1).mean().item())


alphas: [0.3141592741012573, 0.44428831338882446, 0.6283184885978699, 0.8885766267776489, 1.2566369771957397, 1.7771530151367188, 2.5132739543914795, 3.5543060302734375, 5.026547908782959, 7.108611583709717, 10.053094863891602, 14.217223167419434, 20.106189727783203, 28.434444427490234, 40.21237564086914, 56.86888885498047, 80.42475128173828, 113.73777770996094, 160.84950256347656, 227.47552490234375]
K: [[-0.866025447845459, 0.4999999701976776], [1.5730305946703993e-08, -1.0], [0.866025447845459, 0.5]]
K_dagger: [[-0.5773501992225647, 1.0486870039017049e-08, 0.5773502588272095], [0.3333333432674408, -0.666666567325592, 0.3333333134651184]]
torch.Size([2, 5, 20, 3, 1])
AVERAGE ERROR OF ESTIMATED DELTA FROM TRUE DELTA:  1.8650013089427375e-07


Finally, we have a loss function to regularize over $\Delta \theta_j$!

In [19]:
def loss_for_deltas(delta_thetas: torch.Tensor, K_dagger: torch.Tensor, lattice_basis: torch.Tensor, alphas: torch.Tensor):
    deltas = solve_for_deltas(delta_thetas, K_dagger, lattice_basis, alphas)

    # deltas has shape (batch_size, time_steps, J, d)
    return deltas.var(dim=-2).mean()

delta_thetas_noise = delta_thetas + torch.randn_like(delta_thetas)

print("LOSS OF NOISY DELTA_THETAS: ", loss_for_deltas(delta_thetas_noise, K_dagger, lattice_basis, alphas).item())


LOSS OF NOISY DELTA_THETAS:  0.8823283910751343


## Creating and Verifying Location Codes

To work with location codes on a physical space of dimension $d$, we need to be able to create valid locations. In general, our location code is a tensor of shape $(J, d+1, 2)$ where the first dimension is the number of different modules, the second indexes over the vertices of the simplex, and the third contains the $\cos$ and $\sin$ values. To be valid, these last two must satisfy $\cos^2 \theta + \sin^2\theta = 1$, that is, the last dimension must lie on the the unit sphere. But there is a deeper notion of validity as well: the code must have a unique underlying physical interpretation.

To begin, we will need to sample a random location code. One way to do this is to sample a set of phase angles and generate the code from there. Most of the ingredients for this transformation are given above; we've computed the simplex vertices $K$ and the parameters $\alpha_j = \pi\rho^j / s$ for $\rho = \sqrt{2}$. The final piece is the phase $\phi_{i,j}$, which we will define as combination of two factors:
$$
\phi_{i,j} = \tilde{\phi_j} + \xi_i
$$ 
where $\tilde{\phi_j}$ is uniform on $[-\pi, pi)$ and $\xi_i = \frac{2\pi i}{d+1}$. This makes the phases different among all components $i,j$ of the code. In particular, it spaces out the wavevectors $k_i$ evenly over the circle. To say that $\phi_{i,j}$ are the phases assumes the sample represents the encoding of the origin $x = 0$, but this is acceptable because the location $\alpha Kx$ for general $x \neq 0$ would merely represent a simultaneous shift across all components, which can be absorbed into the uniform sample $\tilde{\phi}_i$. So for any set of $n$ samples taken by this method, we can pick one to be the origin and use it to extract a consistent set of phases $\phi_{i,j}$ and consistent physical locations for the other $n-1$ points.

In [58]:
from typing import Tuple

phis = torch.empty((location_dim // 2 // (2 + 1),)).uniform_(-math.pi, math.pi)

def sample(location_dim: int, K: torch.Tensor, alphas: torch.Tensor, phis: torch.Tensor, shape: Tuple[int, ...] = torch.Size()):
    physical_dim = K.shape[1]
    x = torch.randn(shape + (physical_dim,1))
    print(f"sample picked x={x.detach().cpu().numpy().tolist()}")
    while K.ndim < x.ndim:
        K = K[None, ...]
    Kx = (K @ x).squeeze(-1)   # (..., d+1)
    alphas = alphas[..., None] # (J, d+1)
    while alphas.ndim < Kx.ndim:
        alphas = alphas[None, ...]
    aKx = alphas * Kx[..., None, :]  # (..., J, d+1)

    phis = phis[..., None]
    while phis.ndim < aKx.ndim:
        phis = phis[None, ...]

    thetas = aKx + phis # + xis
    return torch.stack([torch.cos(thetas), torch.sin(thetas)], dim=-1).view(shape + (location_dim,))

l1 = sample(120, K, alphas, phis)
l2 = sample(120, K, alphas, phis)

print(f"l1.shape: {l1.shape} code: {l1.detach().cpu().numpy().tolist()}")
print(f"l2.shape: {l2.shape} code: {l2.detach().cpu().numpy().tolist()}")


sample picked x=[[2.4337315559387207], [-1.5284488201141357]]
sample picked x=[[1.5382930040359497], [-0.1443018615245819]]
l1.shape: torch.Size([120]) code: [0.7859589457511902, -0.6182786822319031, 0.7545298933982849, 0.6562657356262207, 0.7913761734962463, 0.6113295555114746, -0.874697208404541, -0.48466983437538147, 0.7772050499916077, -0.6292474269866943, 0.7229195833206177, -0.6909322142601013, 0.9698935747146606, -0.24352917075157166, -0.8122619986534119, 0.5832927823066711, -0.7391322255134583, 0.673560380935669, 0.8373876214027405, 0.5466095209121704, -0.22216443717479706, -0.9750092029571533, -0.37872710824012756, -0.9255083799362183, -0.8994261026382446, -0.4370729327201843, -0.9549795389175415, 0.29667168855667114, -0.8609392046928406, 0.5087078213691711, 0.5464362502098083, 0.8375006914138794, -0.8185031414031982, 0.5745019912719727, -0.5891671180725098, 0.8080111742019653, 0.07016707211732864, 0.997535228729248, 0.9999787211418152, -0.006523387972265482, 0.890897631645202

Now, if we identify $\ell_1$ as the origin $x_1 = 0$, then we can describe $\ell_2$ as the block rotation required to get from $\ell_1$ to $\ell_2$, and from there we can get to the displacement $\Delta x$ such that $x_2 = x_1 + \Delta x$ represents $\ell_2$.

To do this, we use `atan2` and the formula 
$$
\Delta\theta_{ij} = \mathrm{atan2}\left(\left|\eta_{ij} \times \eta_{ij}\right|, \eta_{ij}\cdot\eta_{ij}\right)
$$
which resolves to
$$
\Delta\theta_{ij} = \mathrm{atan2}\left(\cos\theta_1 \sin \theta_2 - \sin\theta_1\cos\theta_2,\,\, \cos\theta_1\cos \theta_2 + \sin\theta_1\sin\theta_2\right)
$$
after which we can call `solve_for_deltas` above.

Note that we will allow for the possibility of codes that have no physical interpretation, which will mean that our results will be of shape $(\ldots, J)$ with $J$ different physical interpretations, one per module. We'll also return the mean of these interpretations. For real physical locations, the variance should be small.

In [63]:
def reshape_to_components(location: torch.Tensor, physical_dim: int):
    shape = tuple(list(location.shape[:-1]) + [-1, physical_dim + 1, 2])

    return location.view(*shape)

def compute_angles(
    location1: torch.Tensor, 
    location2: torch.Tensor, 
    physical_dim: int=2
):
    location1 = reshape_to_components(location1, physical_dim)  # Now has shape (..., J, d+1, 2)
    location2 = reshape_to_components(location2, physical_dim)  # Now has shape (..., J, d+1, 2)

    s2c1 = location2[..., 1] * location1[..., 0].float()
    c2s1 = location2[..., 0] * location1[..., 1].float()
    c2c1 = location2[..., 0] * location1[..., 0].float()
    s2s1 = location2[..., 1] * location1[..., 1].float()
    delta_thetas = torch.atan2(s2c1 - c2s1, c2c1 + s2s1 + 1e-6) # something fishy here

    delta_thetas = delta_thetas.view(delta_thetas.shape[:-2] + (-1,))
    return delta_thetas

def compute_displacements(
    location1: torch.Tensor, 
    location2: torch.Tensor, 
    K_dagger: torch.Tensor, 
    lattice_basis: torch.Tensor, 
    alphas: torch.Tensor, 
):
    physical_dim = K_dagger.shape[0]
    delta_thetas = compute_angles(location1, location2)

    # estimated displacements from thetas, made as small as possible solving across alphas -- but may not agree!
    # deltas has shape (..., J, d)
    deltas = solve_for_deltas(
        delta_thetas, 
        K_dagger, 
        lattice_basis, 
        alphas
    )
    mean_deltas = deltas.mean(dim=-2)

    return deltas.to(location1.dtype), mean_deltas.to(location1.dtype)

delta_thetas = compute_angles(l1, l2, 2)

print(f"delta_thetas.shape: {delta_thetas.shape}")
print(f"delta_thetas: {delta_thetas.detach().cpu().numpy().tolist()}")

deltas, mean_deltas = compute_displacements(l1, l2, K_dagger, lattice_basis, alphas)

print(f"deltas: {deltas.detach().cpu().numpy().tolist()}")
print(f"mean_deltas: {mean_deltas.detach().cpu().numpy().tolist()}")


delta_thetas.shape: torch.Size([60])
delta_thetas: [0.4610426723957062, -0.43484216928482056, -0.026200613006949425, 0.652012825012207, -0.614959716796875, -0.03705320507287979, 0.9220852851867676, -0.8696841597557068, -0.052401237189769745, 1.3040258884429932, -1.2299199104309082, -0.07410615682601929, 1.8441712856292725, -1.73936927318573, -0.10480222851037979, 2.608053207397461, -2.4598400592803955, -0.1482127606868744, -2.5948405265808105, 2.8044447898864746, -0.20960408449172974, -1.0670770406723022, 1.3635026216506958, -0.29642555117607117, 1.0935027599334717, -0.6742949485778809, -0.4192085862159729, -2.134155035018921, 2.7270076274871826, -0.5928510427474976, 2.1870064735412598, -1.348588228225708, -0.8384182453155518, 2.0148725509643555, -0.8291686177253723, -1.1857023239135742, -1.9091696739196777, -2.697178602218628, -1.6768368482589722, -2.2534525394439697, -1.6583366394042969, -2.371405839920044, 2.464839458465576, 0.888831615447998, 2.929509162902832, 1.7762774229049683, 

## A Metric for Fourier Codes

What we are really interested in is a way to compare location codes that respects the structure of the code. We could have used $\|\ell - \ell'\|$, which is defined on $[1, -1]^L$, but the structure of a fourier location code means that the different modules $j$ have different wavelengths $\alpha_j$, which means that different components should have different weights. Since we've sorted our wavelengths, we end up with the early dimensions mattering much more than the later dimensions. We can leverage the foregoing formulae to compose a pseudometric over location codes by extracting the block rotations that make two location align, extracting $\Delta \theta_{ij}$, computing $\Delta x$ and finally return $\|\Delta x\|$ as our norm. In other words, we return the physical displacement underlying the two codes. In terms of codes, I believe it is a full metric; but in terms of underlying physical space it is a pseudometric because finite-dimensional location codes cannot make distinctions that are too large or too small. 

In addition to the distance, we can consider the case where a "location code" has varying physical interpretations across modules $j$. In this case, we can compute a mean physical interpretation and add the variance to our distance as well.

In [68]:
def pseudo_distance(
    location1: torch.Tensor, 
    location2: torch.Tensor, 
    K_dagger: torch.Tensor, 
    lattice_basis: torch.Tensor, 
    alphas: torch.Tensor, 
    squared: bool=False, 
    use_variance: bool=True
):
    deltas, mean_deltas = compute_displacements(location1, location2, K_dagger, lattice_basis, alphas)
    J = deltas.shape[-2]
    assert J > 1, "J must be greater than 1"

    squared_distances = mean_deltas.square().sum(dim=-1)

    if use_variance:
        dev_deltas = deltas - mean_deltas[..., None, :]
        variances = dev_deltas.square().sum(dim=-1).mean(dim=-1) * (J / (J - 1))  # (...)
        
        final_squared_distances = squared_distances + variances

    else:
        final_squared_distances = squared_distances

    if squared:
        return final_squared_distances
    else:
        return (final_squared_distances + 1e-12).sqrt()


dist = pseudo_distance(l1, l2, K_dagger, lattice_basis, alphas)
print(f"dist: {dist.detach().cpu().numpy().tolist()}")

dist: 1.650359034538269


Now, when implementing TEM variants, what we are interested in is not just the distance but the cross-distance among many items. That is, given $\{\ell_i\}$ and $\{\ell_j\}we want to compute $\{\|\ell_i - \ell_j\|\}$ for all $i,j$. We are interested in the case where $\ell_i$ has shape $(B, T, L)$ and $\ell_j$ has shape $(B, S, L)$, and we will limit the flexibility for this case. This is easily done with shape tricks:

In [73]:
def cross_distance(location1: torch.Tensor, location2: torch.Tensor, K_dagger: torch.Tensor, lattice_basis: torch.Tensor, alphas: torch.Tensor, squared: bool=False):
    return pseudo_distance(location1[..., :, None, :], location2[..., None, :, :], K_dagger, lattice_basis, alphas, squared=squared)

B = 2
T = 10
S = 8
li = sample(120, K, alphas, phis, (B, T))
lj = sample(120, K, alphas, phis, (B, S))

cross_dist = cross_distance(li, lj, K_dagger, lattice_basis, alphas)
print(f"cross_dist: {cross_dist}")
print(f"Min dist = {cross_dist.min().item()} Mean dist = {cross_dist.mean().item()} Max dist = {cross_dist.max().item()}")


sample picked x=[[[[-1.1615135669708252], [-0.7597837448120117]], [[0.35750439763069153], [1.1878219842910767]], [[-0.19153252243995667], [-1.5257058143615723]], [[0.6116032004356384], [-1.125410795211792]], [[2.4426684379577637], [-0.8654753565788269]], [[0.9214881062507629], [-0.10546930879354477]], [[0.5841965675354004], [-1.245393991470337]], [[-0.5600711703300476], [-1.2647414207458496]], [[-0.8346246480941772], [-0.3636777997016907]], [[-0.5528364181518555], [0.3580321967601776]]], [[[-0.1623544692993164], [-0.6323263049125671]], [[-0.9357377290725708], [0.7213431596755981]], [[1.4950803518295288], [0.4351041913032532]], [[-0.04706111177802086], [0.8739541172981262]], [[-0.5042762160301208], [-0.3886164128780365]], [[1.186394214630127], [-0.08738738298416138]], [[0.42632028460502625], [-0.3975406885147095]], [[-0.48055240511894226], [1.2895159721374512]], [[-0.7245559096336365], [-2.5134994983673096]], [[-0.7143908739089966], [-0.29565849900245667]]]]
sample picked x=[[[[-3.42521

## Class Implementation and Relative Sampling

This code is implemented in `tree_world.models.fourier_metric.FourierMetric` with $K$, $K^\dagger$, $\Lambda$, $\alpha$, and $\phi$ as buffer variables. It is implemented in a `torch.nn.Module` but has does not generally have learnable parameters. This class can be used to sample and compare location codes.

In [75]:
from tree_world.models.fourier_metric import FourierMetric

metric = FourierMetric(location_dim=120, dim=2, scale=10.0, ratio=math.sqrt(2.0))

li = metric.sample((B, T))
lj = metric.sample((B, S))
dist = metric.cross_distance(li, lj)
print(f"dist: {dist}")


dist: tensor([[[0.7840, 1.1564, 1.7621, 0.3508, 1.9430, 2.9717, 2.3342, 2.0085],
         [0.1863, 1.5682, 2.1443, 0.8113, 2.2612, 2.3728, 2.9172, 1.1406],
         [2.2031, 0.8214, 0.4500, 1.6356, 0.8010, 3.0539, 0.8011, 2.8022],
         [1.0527, 1.2179, 1.7971, 0.4862, 2.0194, 3.1952, 2.1893, 2.2433],
         [1.2312, 0.7507, 0.8253, 1.1500, 0.7932, 1.8259, 2.0155, 1.6132],
         [1.0813, 2.3730, 3.0053, 1.5921, 3.1124, 2.9336, 3.6309, 1.4181],
         [2.6152, 1.9943, 1.6945, 2.6432, 1.3622, 1.5236, 3.0077, 2.5114],
         [2.1405, 0.7594, 0.5768, 1.5572, 0.8878, 3.1462, 0.7382, 2.6731],
         [0.8370, 2.0691, 2.5758, 1.5500, 2.5066, 2.0312, 3.4308, 0.4575],
         [1.7941, 2.0104, 1.8470, 2.1093, 1.6192, 0.6577, 3.1241, 1.3045]],

        [[0.5561, 1.6033, 1.5297, 1.0337, 0.9776, 2.4080, 0.8772, 1.2853],
         [0.8804, 1.4137, 0.2710, 2.2135, 1.0666, 2.9678, 0.9805, 0.0462],
         [1.4350, 0.5675, 1.4746, 2.1287, 1.8249, 1.5657, 0.3888, 1.3452],
         [3.6345,

The module `tree_world.models.fourier_metric` also defines a relative sampler in support of our TEM VAE. To understand this problem, suppose you have a reference location $\ell_0$, and you want to sample a neighboring location $\ell$ that preferentially lies in some neighborhood of scale $\epsilon > 0$. Again, we can revert to physical space for our operations. We can sample $\Delta x \sim \mathcal{N}\left(0, \epsilon^2 I\right)$ and then compute $\Delta\theta_{ij}$ and the block rotation $R(\Delta\theta_{ij})$. Applying this rotation to $\ell_0$ yields a location that is $\epsilon$-close to $\ell_0$ in terms of the pseudometric defined above. Here is the code:

In [89]:
import torch.distributions as D
from typing import Optional, Tuple

class FourierCodeDistribution(D.Distribution):
    support = D.constraints.real
    has_rsample = True

    def __init__(self, metric: FourierMetric, reference_location: torch.Tensor, scale: torch.Tensor, 
                 batch_lengths: Optional[torch.Tensor]=None,idx: Optional[torch.Tensor]=None, validate_args=None):
        self.metric = metric
        self.reference_location = reference_location
        self.batch_lengths = batch_lengths
        self.idx = idx
        self.dtype = reference_location.dtype
        self.device = reference_location.device
        self.scale = scale

        batch_shape = reference_location.shape[:-1]

        assert scale.shape == () or scale.shape == batch_shape
        if scale.shape == ():
            self.scale = self.scale.expand(batch_shape)

        # jac_weights will have shape (..., d+1)
        reference_location = self.metric.reshape_to_components(reference_location)  # Now has shape (..., J, d+1, 2)
        alphas = self.metric.alphas
        while alphas.ndim < reference_location.ndim - 1:
            alphas = alphas[None, ...]
        alphas = alphas.transpose(-2, -1) # (..., J, d+1)
        jac_weights = (alphas.square() * reference_location.square().sum(dim=-1)).sum(dim=-2)

        # K is (d+1, d); need (wK)^T K = (..., d+1, d+1)
        K = self.metric.K
        while K.ndim < jac_weights.ndim + 1:
            K = K[None, ...]
        wK = (jac_weights[..., None] * K)
        jac_squared = K.transpose(-2, -1) @ wK

        eps = 1e-6
        I = torch.eye(self.metric.dim, device=jac_squared.device, dtype=torch.float32)
        while I.ndim < jac_squared.ndim:
            I = I[None, ...]
        jac_squared = jac_squared.float() + eps * I
        logdet = torch.logdet(jac_squared)
        self.logdet_jac = - 0.5 * logdet.to(self.dtype)

        super().__init__(batch_shape=batch_shape, event_shape=(self.metric.location_dim,), validate_args=validate_args)

    def sample(self, sample_shape: Tuple[int, ...] = torch.Size()):
        # sample Delta x ~ N(0, I_M)
        u = torch.randn(sample_shape + self.batch_shape + (self.metric.dim,), device=self.device, dtype=self.dtype)
        scale = self.scale[..., None]
        while scale.ndim < u.ndim:
            scale = scale[None, ...]
        deltas = scale * u
        print(f"deltas: {deltas.shape} reference_location: {self.reference_location.shape}")
        locations = self.metric.apply_displacement(deltas, self.reference_location)
        return locations, deltas
    
    rsample = sample

    def log_prob(self, locations: torch.Tensor, displacements: Optional[torch.Tensor]=None):
        if displacements is None:
            _, displacements = self.metric.compute_displacements(self.reference_location, locations)

        scale = self.scale[..., None]
        while scale.ndim < displacements.ndim:
            scale = scale[None, ...]

        displacements = displacements / scale

        gaussian_log_prob = (
            - 0.5 * displacements.square().sum(dim=-1)
            - self.metric.dim * (0.5 * math.log(2.0 * math.pi) + torch.log(self.scale))
        )

        return gaussian_log_prob + self.logdet_jac

And, a demonstration:

In [98]:
l0 = metric.sample((B, T))
epsilon = torch.empty(B, T).uniform_(0.01, 0.1)

sampler = FourierCodeDistribution(metric, l0, epsilon)
l, deltas = sampler.sample()
print(f"l: {l.shape}")
dist = metric.pseudo_distance(l, l0)
print(f"Dist: {dist}")
print(f"Min dist = {dist.min().item()} Mean dist = {dist.mean().item()} Max dist = {dist.max().item()}")
print(f"Epsilon-scaled dist: {dist / epsilon}")
print(f"Epsilon-scaled dist min = {(dist / epsilon).min().item()} mean = {(dist / epsilon).mean().item()} max = {(dist / epsilon).max().item()}")



deltas: torch.Size([2, 10, 2]) reference_location: torch.Size([2, 10, 120])
l: torch.Size([2, 10, 120])
Dist: tensor([[0.0663, 0.3002, 0.0385, 0.0116, 0.0170, 0.0176, 0.0388, 0.0769, 0.0485,
         0.0557],
        [0.0752, 0.0259, 0.0315, 0.0055, 0.0260, 0.0182, 0.0364, 0.0156, 0.1820,
         0.0102]])
Min dist = 0.005459532141685486 Mean dist = 0.05486201122403145 Max dist = 0.3001953065395355
Epsilon-scaled dist: tensor([[0.7807, 3.2476, 0.9159, 0.1926, 0.5171, 0.3641, 0.9831, 1.3824, 0.9509,
         0.9069],
        [1.8777, 0.6585, 0.3179, 0.5015, 0.3769, 0.2366, 1.5025, 0.2116, 2.4488,
         0.4110]])
Epsilon-scaled dist min = 0.19261732697486877 mean = 0.9392115473747253 max = 3.247570753097534


/Users/alockett/dev/tree-world/venv/lib/python3.12/site-packages/torch/distributions/distribution.py:62: UserWarning: <class '__main__.FourierCodeDistribution'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


As we see, our referential sampler provides samples that are "close" to the reference point, and when we scale for epsilon, the mean epsilon-scaled distance is about $1$, as we expect.

The one bit that hasn't been explained is the log probability calculation, which is needed for the VAE loss on TEM. But that is a longer discussion that will be more easily addressed after defining an embedding-based pseudo-metric, which we'll do next.

# Conclusion

We've built a metric for Fourier location codes that respects the underlying physicality. Together with the embedding pseudo-metric we'll build next, we can use this metric to implement an advanced spatial memory for TEM-t .